In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [3]:
# Reload dataset
credit = fetch_openml(name='credit-g', version=1, as_frame=True)

df = credit.data
df['target'] = credit.target

# Convert target to binary (1 = bad, 0 = good)
df['target'] = df['target'].map({'good': 0, 'bad': 1})

print("Dataset loaded!")
print("Shape:", df.shape)
print("\nTarget distribution:")
print(df['target'].value_counts())

Dataset loaded!
Shape: (1000, 21)

Target distribution:
target
0    700
1    300
Name: count, dtype: int64


In [4]:
# Separate categorical and numerical columns
categorical_cols = ['checking_status', 'credit_history', 'purpose', 
                    'savings_status', 'employment', 'personal_status',
                    'other_parties', 'property_magnitude', 'other_payment_plans',
                    'housing', 'job', 'own_telephone', 'foreign_worker']

numerical_cols = ['duration', 'credit_amount', 'installment_commitment', 
                  'residence_since', 'age', 'existing_credits', 'num_dependents']

# Encoding categorical columns
le = LabelEncoder()
df_encoded = df.copy()

for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

print("Categorical columns encoded!")
print("\nFirst 5 rows:")
df_encoded.head()

Categorical columns encoded!

First 5 rows:


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,target
0,1,6,1,6,1169,4,3,4,3,2,...,3,67,1,1,2,1,1,1,1,0
1,0,48,3,6,5951,2,0,2,0,2,...,3,22,1,1,1,1,1,0,1,1
2,3,12,1,2,2096,2,1,2,3,2,...,3,49,1,1,1,3,2,0,1,0
3,1,42,3,3,7882,2,1,2,3,1,...,1,45,1,0,1,1,2,0,1,0
4,1,24,2,4,4870,2,0,3,3,2,...,2,53,1,0,2,1,2,0,1,1


In [5]:
# Scale numerical features
scaler = StandardScaler()

df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

print("Numerical features scaled!")
print("\nScaled numerical features stats:")
df_encoded[numerical_cols].describe().round(2)

Numerical features scaled!

Scaled numerical features stats:


,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,0.00,0.00,0.00,-0.00,0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.40,-1.07,-1.76,-1.67,-1.46,-0.70,-0.43
25%,-0.74,-0.68,-0.87,-0.77,-0.75,-0.70,-0.43
50%,-0.24,-0.34,0.02,0.14,-0.22,-0.70,-0.43
75%,0.26,0.25,0.92,1.05,0.57,1.03,-0.43
max,4.24,5.37,0.92,1.05,3.47,4.49,2.33


In [6]:
# Split Dataset
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']

print("Before SMOTE:")
print(y.value_counts())

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("\nAfter SMOTE:")
print(pd.Series(y_resampled).value_counts())
print("\nNew dataset shape:", X_resampled.shape)

Before SMOTE:
target
0    700
1    300
Name: count, dtype: int64

After SMOTE:
target
1    700
0    700
Name: count, dtype: int64

New dataset shape: (1400, 20)


In [7]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(
                                    X_resampled, y_resampled, 
                                    test_size=0.2, 
                                    random_state=42,
                                    stratify=y_resampled)

In [8]:
import os

# Create data folder if not exists
os.makedirs('../data', exist_ok=True)

# Save processed data
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
pd.Series(y_train).to_csv('../data/y_train.csv', index=False)
pd.Series(y_test).to_csv('../data/y_test.csv', index=False)

# Save feature names for later use
feature_names = X.columns.tolist()
pd.Series(feature_names).to_csv('../data/feature_names.csv', index=False)

print("All files saved successfully!")
print("\nSaved files:")
print("- X_train.csv:", X_train.shape)
print("- X_test.csv:", X_test.shape)
print("- y_train.csv:", y_train.shape)
print("- y_test.csv:", y_test.shape)
print("- feature_names.csv:", len(feature_names), "features")

All files saved successfully!

Saved files:
- X_train.csv: (1120, 20)
- X_test.csv: (280, 20)
- y_train.csv: (1120,)
- y_test.csv: (280,)
- feature_names.csv: 20 features
